In [ ]:
from ase import Atoms
from ase.io import read
from ase.units import Ha
from ase_dirac import DIRAC

Note that the current interface is rather minimal. We plan to add more functionality in next releases but of course also welcome user contributions (simply make a pull request if you have added something useful to this interface).

---

# Input definition and calculator constructor methods
In the previous (part of the) Tutorial, we defined the calculation setup using the internal dictionary representation of the `DIRAC` calculator object. In this tutorial we will show a small addition to abstract that layer when diract input files are already availiable. 
##  Direct construction
The first "layer" is the construction of an ASE `Atoms` object with the calculator defined, ready to interact with the functionalities of ASE. For this we define two files `molecule.xyz` and `dirac_example.inp`:

In [ ]:
xyz_string = """2
F2
O      0 0 0
O      0.00000       -0.75545     -0.47116
"""

fscc_str = """ **DIRAC
    .TITLE
    Molecular Te. DEA from the dianion.
    .WAVE F
    **HAMILTONIAN
    .LVCORR
    **WAVE FUNCTIONS
    .SCF
    .RELCCSD
    ! because we want this
    *SCF
    .PRINT 
    1
    !.CLOSED SHELL
    !54 52
    **RELCCSD
    .ENERGY
    
    
    .PRINT
    1
    .FOCKSP
    *CCFSPC
    .DOIE2
    .NACTH
    1 1 0 0
    .MAXIT
    200
    **MOLECULE
    *BASIS
    .DEFAULT
    dyall.3zp
    ! **Moltra
    ! .Active
    ! energy -10.0 20.0 1.0
    *END OF
"""

with open("molecule.xyz", "w") as f:
    f.write(xyz_string)

with open("dirac_example.inp", "w") as f:
    f.write(fscc_str)

We can now define an atoms object from an input file and a coordinate file as:

In [ ]:
molecule = DIRAC.ASE_atoms_from_inpfile_and_coords("dirac_example.inp", "molecule.xyz")
print(molecule.calc.parameters)

And run the calculations as in the previous (part of the) tutorial:

In [ ]:
molecule.get_potential_energy() / Ha

Making possible to run whole DIRAC calculations with only two lines of code. 

Also, the previous molecule object can be obtained with a predefined `Atoms` object if necesssary (for example if some previous interaction via ASE as been required): 

In [ ]:
molecule_2 = read("molecule.xyz", format="xyz")
molecule_2 = DIRAC.ASE_atoms_from_inpfile("dirac_example.inp", molecule_2)
molecule_2.get_potential_energy() / Ha

Which can be convinient given the case that input files and/or coordinate files are already available. 

---

## Calculator constructors
The previous definition is very simple, but under the hood the following takes place. ASE's `Atoms` requires a `calculator` object to run calculations. In our case we use the custom `DIRAC` calculator object. Therefore, it is necessary to initialize this object and attach it to the `Atoms`. Once the file (or string) are defined, it is possible to construct the object as:

```python
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.from_file("dirac_example.inp")
```

Or :

```python
ase_molecule.calc = DIRAC.from_string(fscc_str)
```


In [ ]:
ase_molecule = read("molecule.xyz", format="xyz")
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

# Also it can be called from string as:
# ase_molecule.calc = DIRAC.calculator_from_string(fscc_str)

Where we can see the internal dictionary definition that this calculator presents:

In [ ]:
param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

However, the default options in the `DIRAC.check_dirac_attributes()` method conflict with either inputted keywords or default `pam` parsing/default execution:
- `*charge: {'.charge': '0'}`: Occupation was established with `.CLOSED SHELL \n 54 52`. For the diatomic $O_2$ as proposed, the correct charge would be $-90$. This is not a big deal as occupation overrides charge. However this is redundant as the DIRAC parsing subroutine will default to $0$ in the absence of keyword.
- `*symmetry: {'.nosym': '#'}`: Symmetry was implicitly defined with `.CLOSED SHELL \n 54 52`. However, it is overriden, but `pam` would try to detect symmetry. This is a big deal as symmetry is ignored. Also, as it is currently, since `.nosym` is by default and there is not a keyword to activate automatic symmetry, all systems in which `.linear` is not applicable will be run without symmetry. 
- `integrals, {'.nucmod': '1'}` and `moltra, {'.active': 'all'}`: `moltra, {'.active': 'all'}` directly contradicts the default `MOLTRA` behaviour with `energy -10.0 20.0 1.0`. This can present issues in the correlation calculations, where the recalculation of the energy does not lead to the same values. 

Therefore, it might be possible to align the calculator default behaviour with `pam`'s default behaviour.

Which is not ideal. However, with these constructor methods, it is possible to start from a `.inp` file and just modify certain keyword-value pairs to override the defaults:

In [ ]:
ase_molecule.calc = DIRAC.calculator_from_file("dirac_example.inp")

ase_molecule.calc.parameters.moltra.update({".active": "energy -10 20 1"})
ase_molecule.calc.parameters.molecule.update({"*symmetry": {".LINEAR": ""}})

param_list = [
    f"{key}, {ase_molecule.calc.parameters[key]}\n"
    for key in ase_molecule.calc.parameters.keys()
]

print("".join(param_list))

try:
    ase_molecule.calc.calculate()
except TypeError:
    ase_molecule.get_potential_energy() / Ha


---
# Calculation results and I/O
## Energies and direct parsing
In the explainatory notebook `ASETutorial` all the calls to compute the energy are later divided by the constant `Ha`:

```python
energy.append(ase_molecule.get_potential_energy() / Ha)
```

However, the energies are multiplied by the same constant when parsing the output and adding the energy to results:

```python
with open(output_filename, "r") as file:
        content = file.read() 

    ehfmatch = re.search(_scf_energy_re, content)
    if ehfmatch is not None:
        energy = float(ehfmatch.group(1)) * Ha # Here the energy is multiplied by Ha 
```

Maybe there is an underlying reason for this, but the multiplication to then divide by the same constant might be redundant. 



---

Regarding the extraction of values, currently only the highest correlated energy is saved to the `calc.results` dictionary. However, it might be of interest to also keep the scf or MP2 energies from the parsed values. Therefore a small change was added in `dirac_io.py` on the `read_dirac_out` function to keep them with their own labels:

```python 
    output = {}
    energy = None

    with open(output_filename, 'r') as file:
        content = file.read()  # Read the entire content of the file

    ehfmatch = re.search(_scf_energy_re, content)
    if ehfmatch is not None:
        energy = float(ehfmatch.group(1)) * Ha
        output.update({'SCF_energy':energy}) # this way each energy is saved with the corresponding keyword

    # [...] 

    output.update({'energy': energy}) # but the final energy logic is the same and calc.parameters['energy'] will be the most correlated energy
```


---

## h5 functionality from the calculator 
On the result data and I/O, the read results method in the calculator was extended to retrieve some data from the `.h5` file obtained in the calculation: 

```python
def read_results(self): 
    """Read energy from DIRAC output file. Try to extract from h5 if availiable"""
    out_file = self.label + ".out"
    output = read_dirac_out(out_file)
    self.results = output
    self._extract_common_results_from_h5()
```

Where `_extract_common_results_from_h5()` extracts from the associated `.h5` file what now is an arbitrary list of (probably) values of interest: 

```python
        commonly_requested_results: dict = {
            "result/wavefunctions/scf/energy": "scf_energy",
            "result/wavefunctions/scf/mobasis/eigenvalues": 'scf_eigenvalues',
        }
```

This way, when calling for `ase_molecule.calc.results`, we can obtain more than just the final energy:

In [ ]:
ase_molecule.calc.results

Also, a property `DIRAC.h5_contents` property was added to list all availiable properties in the `.h5` file: 

``` python
    @property
    def h5_contents(self) -> list:
```

In [ ]:
ase_molecule.calc.h5_contents[0:10]

And a `.extract_h5_value` method was added in order to extract properties by key:

In [ ]:
n_ao = ase_molecule.calc.extract_h5_value("result/operators/ao_matrices/BETAMAT FFFT")
print(repr(n_ao))

Where of course error handling was introduced for missing `.h5` files or wrong key values. 